## Read data

demonstration data set from the UEA collection

In [ ]:
import numpy as np
import time


def load_data(file_path):
    data = np.load(file_path, allow_pickle=True).item()
    X_train = data["train"]["X"]
    y_train = np.array([int(x) for x in data["train"]["y"]])
    X_test = data["test"]["X"]
    y_test = np.array([int(x) for x in data["test"]["y"]])
    return X_train, y_train, X_test, y_test


X_train, y_train, X_test, y_test = load_data("CounterMovementJump.npy")
print("X_train dims: ", X_train.shape)
print("X_test dims: ", X_test.shape)

X_train dims:  (1426, 50, 161)
X_test dims:  (595, 50, 161)


your X_train and X_test should be of the shape `(n_samples, n_channels, seq_len=512)` to apply the foundation model

if original sequence length is different, resize it, for example, using the following function:

In [2]:
import torch
import torch.nn.functional as F


def resize(X):
    X_scaled = F.interpolate(
        torch.tensor(X, dtype=torch.float), size=512, mode="linear", align_corners=False
    )
    return X_scaled.numpy()


X_train, X_test = resize(X_train), resize(X_test)

print("X_train dims: ", X_train.shape)
print("X_test dims: ", X_test.shape)

X_train dims:  (1426, 50, 512)
X_test dims:  (595, 50, 512)


## Load model

In [3]:
from mantis.architecture import Mantis8M

device = "cpu"  # set device
network = Mantis8M(device=device)  # init model
network = network.from_pretrained("paris-noah/Mantis-8M")  # load weights

/opt/homebrew/Caskroom/miniconda/base/envs/mantis/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Extract deep features without an adapter

By default, `.transform()` extract features for each channel independently and concatenate all the outputs in a flattened manner.

The same goes for `.fit()` method, when fine-tuning is performed.

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import RidgeClassifier
from mantis.trainer import MantisTrainer

transformation_start = time.time()
model = MantisTrainer(device=device, network=network)  # init trainer
Z_train = model.transform(X_train)
Z_test = model.transform(X_test)
trabsformation_end = time.time()
print(f"Transformation took {trabsformation_end - transformation_start} seconds")
classifier_start = time.time()
predictor = RandomForestClassifier(n_estimators=100, random_state=42)
predictor.fit(Z_train, y_train)
classifier_end = time.time()
print(f"Random Forest fit {classifier_end - classifier_start} seconds")
prediction_start = time.time()
y_pred = predictor.predict(Z_test)
prediction_end = time.time()
print(f"Random Forest Prediction took {prediction_end - prediction_start} seconds")
print(f"Accuracy for random forest {np.mean(y_test == y_pred)}")


classifier_start = time.time()
predictor = LogisticRegression(max_iter=1000, random_state=42)
predictor.fit(Z_train, y_train)
classifier_end = time.time()
print(f"Logistic regression fit {classifier_end - classifier_start} seconds")
prediction_start = time.time()
y_pred = predictor.predict(Z_test)
prediction_end = time.time()
print(
    f"Logistic regression Prediction took {prediction_end - prediction_start} seconds"
)
print(f"Accuracy for logistic regression {np.mean(y_test == y_pred)}")

classifier_start = time.time()
predictor = RidgeClassifier(random_state=42)
predictor.fit(Z_train, y_train)
classifier_end = time.time()
print(f"Ridge Classifier fit {classifier_end - classifier_start} seconds")
prediction_start = time.time()
y_pred = predictor.predict(Z_test)
prediction_end = time.time()
print(f"Ridge Classifier Prediction took {prediction_end - prediction_start} seconds")
print(f"Accuracy for Ridge Classifier {np.mean(y_test == y_pred)}")

Transformation took 108.44016194343567 seconds
Random Forest fit 4.5706610679626465 seconds
Random Forest Prediction took 0.00902700424194336 seconds
Accuracy for random forest 0.692436974789916
Logistic regression fit 2.212432861328125 seconds
Logistic regression Prediction took 0.013662099838256836 seconds
Accuracy for logistic regression 0.6857142857142857
Ridge Classifier fit 0.12549233436584473 seconds
Ridge Classifier Prediction took 0.0014998912811279297 seconds
Accuracy for Ridge Classifier 0.6235294117647059


## Extract deep features with an adapter

If the number of channels is too large, one approach is to reduce the dimensionality first, then pass the transformed input to the foundation model. This resembles a dimensionality reduction problem, with two key nuances:

 1. The data is three-dimensional.
 2. The dimensionality reduction algorithm should preserve the temporal patterns, as these are important for the foundation model.
 
In this package, we provide several simple solutions to address these challenges, while leaving the development of more sophisticated and efficient solutions for future work.

### Dimension reduction along channels

Given a 3d dataset of size $(N, D, T)$, an intuitive approach would be to reshape it to $(N, D \times T)$ in order to apply classical dimension reduction algorithms like PCA. However, this may disrupt the temporal structure as new features will be combine different time stamps and channels at the same.

Therefore, we reshape the data to $(N \times T, D)$, allowing the dimension reduction algorithm to focus on correlations between channels over all time steps, effectively capturing spatial correlations while preserving temporal information. In the case of linear feature transformation, we eventually learn a rotation matrix $W \in \mathbb{R}^{D' \times D}$ that linearly combines the original $D$ channels into new $D'$ channels, which intuitevely allows to preserve most of the temporal patterns.

We have implemented this approach as a wrapper called `MultichannelProjector` that supports any 2d unsupervised dimension reduction algorithm that follows the `scikit-learn` convention with a `n_components` argument as well as `fit` and `transform` methods.

In [5]:
from mantis.adapters import MultichannelProjector
from sklearn.decomposition import PCA

adapter = MultichannelProjector(new_num_channels=5, base_projector=PCA)

We also provided shortcuts for 3 algorithms: `PCA`, `TruncatedSVD` from `sklearn.decomposition` and `SparseRandomProjection` from `sklearn.random_projection`.

In [6]:
# for PCA
adapter = MultichannelProjector(new_num_channels=5, base_projector="pca")
# for TruncatedSVD
adapter = MultichannelProjector(new_num_channels=5, base_projector="svd")
# for SparseRandomProjection
adapter = MultichannelProjector(new_num_channels=5, base_projector="rand")

### Patch-wise dimension reduction

As previous approach completely ignores interdependecies between the measurements at different timestamps, one simple approach to overcome this issue is to split the time dimension into non-overlapping patches. In other words, we reshape $(N, D, T)$ data into $(N \times P, S \times D)$, where $P$ is the number of patches, $S$ is the patch size, $T = S \times D$, and then use a dimension reduction algorithm, which results in a projected data matrix of size $(N \times P, S \times D')$. Finally, we reshape the transformed input to $(N, D', T)$.

This approach we implemented through an optional argument `patch_window_size` in `MultichannelProjector`.

In [7]:
from mantis.adapters import MultichannelProjector

adapter = MultichannelProjector(
    new_num_channels=5, base_projector="pca", patch_window_size=8
)

### Channel selection

Alternatively to dimension reduction, we can use feature selection approaches to reduce the number of channels. We have implemented a very simple approach where we reshape data to $(N \times T, D)$, then sort the channels by variance in descending order and keep $D'$ first ones.

In [8]:
from mantis.adapters import VarianceBasedSelector

adapter = VarianceBasedSelector(new_num_channels=5)

### Optional: scaling channels

This is step is optional and depends on an application. If all channels are measured in different units, it makes sense to first scale channels before applying an adapter. 
If all channels are measured in the same units, scaling is not necessary and sometimes can be even harmful as it may change the channel importance ranking.

In [9]:
# scale training and test data
if False:
    from sklearn.preprocessing import StandardScaler

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(
        X_train.shape
    )
    X_test = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(
        X_test.shape
    )

### Apply an adapter

For demonstration, we have chosen patched PCA and no channel scaling.

In [10]:
from mantis.adapters import MultichannelProjector

adapter = MultichannelProjector(
    new_num_channels=2, base_projector="pca", patch_window_size=8
)

All these adapters follow the same pipeline which is written as follows: 

In [11]:
adapter.fit(X_train)
X_reduced_train, X_reduced_test = adapter.transform(X_train), adapter.transform(X_test)

print("X_reduced_train dims: ", X_reduced_train.shape)
print("X_reduced_test dims: ", X_reduced_test.shape)

X_reduced_train dims:  (1426, 2, 512)
X_reduced_test dims:  (595, 2, 512)


Extract deep features, learn a classifier and evaluate the perfomance

In [12]:
from sklearn.ensemble import RandomForestClassifier
from mantis.trainer import MantisTrainer

transformation_start = time.time()
model = MantisTrainer(device=device, network=network)  # init trainer
Z_train = model.transform(X_reduced_train)
Z_test = model.transform(X_reduced_test)
transformation_end = time.time()
print(f"Transformation took {transformation_end - transformation_start} seconds")

Transformation took 4.834660053253174 seconds


In [13]:
classifier_start = time.time()
predictor = RandomForestClassifier(n_estimators=100, random_state=42)
predictor.fit(Z_train, y_train)
classifier_end = time.time()
print(f"Random Forest fit {classifier_end - classifier_start} seconds")
prediction_start = time.time()
y_pred = predictor.predict(Z_test)
prediction_end = time.time()
print(f"Random Forest Prediction took {prediction_end - prediction_start} seconds")
print(f"Accuracy for random forest {np.mean(y_test == y_pred)}")


classifier_start = time.time()
predictor = LogisticRegression(max_iter=1000, random_state=42)
predictor.fit(Z_train, y_train)
classifier_end = time.time()
print(f"Logistic regression fit {classifier_end - classifier_start} seconds")
prediction_start = time.time()
y_pred = predictor.predict(Z_test)
prediction_end = time.time()
print(
    f"Logistic regression Prediction took {prediction_end - prediction_start} seconds"
)
print(f"Accuracy for logistic regression {np.mean(y_test == y_pred)}")

classifier_start = time.time()
predictor = RidgeClassifier(random_state=42)
predictor.fit(Z_train, y_train)
classifier_end = time.time()
print(f"Ridge Classifier fit {classifier_end - classifier_start} seconds")
prediction_start = time.time()
y_pred = predictor.predict(Z_test)
prediction_end = time.time()
print(f"Ridge Classifier Prediction took {prediction_end - prediction_start} seconds")
print(f"Accuracy for Ridge Classifier {np.mean(y_test == y_pred)}")

Random Forest fit 1.1680059432983398 seconds
Random Forest Prediction took 0.005234956741333008 seconds
Accuracy for random forest 0.34789915966386553
Logistic regression fit 1.0435540676116943 seconds
Logistic regression Prediction took 0.0024340152740478516 seconds
Accuracy for logistic regression 0.33613445378151263
Ridge Classifier fit 0.007588863372802734 seconds
Ridge Classifier Prediction took 0.00021028518676757812 seconds
Accuracy for Ridge Classifier 0.334453781512605


/opt/homebrew/Caskroom/miniconda/base/envs/mantis/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In our demonstration example, the performance of "patched PCA -> Mantis -> RF" is even better than  "Mantis -> RF", which may be explained by the fact that with PCA we can learn new disetangled channels which incorporate interactions between the original channels. [Benechehab et al. (2024)](https://arxiv.org/abs/2410.11711) showed that using such an adapter, one can consistently improve the multichannel forecasting quality, but in the classification context, we have found that it is rather dataset-dependent. Our main motivation of adding adapters was to reduce the memory and time complexity, which can be important if full fine-tuning of the foundation model is required.

## Differentiable adapter for fine-tuning

Previously, we demonstrated the use of standalone adapters, which can be applied before passing the input to the foundation model. In this section, we show how to use a differentiable adapter, implemented as a pytorch module, that is learned through backpropagation as part of the overall network.

The advantage of such an adapter is that the dimensionality reduction layer is optimized according to the ultimate classification loss. However, this approach is significantly slower, as the adapter and classification head are learned jointly, requiring a forward pass through the foundation model for every optimization step — even if the foundation model's weights are frozen.

We have implemented a simple adapter that applies a learnable rotation matrix $W \in \mathbb{R}^{D' \times D}$, linearly combining original $D$ channels into new $D'$ channels. 

In [14]:
from mantis.adapters import LinearChannelCombiner

adapter = LinearChannelCombiner(num_channels=X_train.shape[1], new_num_channels=5)

Alternatively, you can define your own pytorch module:

In [15]:
from torch import nn


class MyAdapter(nn.Module):
    def __init__(self, num_channels, new_num_channels):
        super().__init__()
        self.num_channels = num_channels
        self.new_num_channels = new_num_channels

    def forward(self, x):
        raise NotImplementedError

### Adapter + head fine-tuning 

In [16]:
fine_tuning_type = "adapter_head"
finetuning_start = time.time()
# fine-tune the model
model.fit(
    X_train, y_train, num_epochs=100, fine_tuning_type=fine_tuning_type, adapter=adapter
)
finetuning_end = time.time()
print(f"Fine-tuning took {finetuning_end - finetuning_start} seconds")
prediction_start = time.time()
# predict labels
y_pred = model.predict(X_test)
prediction_end = time.time()
print(f"Prediction took {prediction_end - prediction_start} seconds")
# evaluate performance
print(f"Accuracy on the test set is {np.mean(y_test == y_pred)}")

  0%|          | 0/100 [00:04<?, ?it/s]


KeyboardInterrupt: 

### Full fine-tuning

In [ ]:
fine_tuning_type = "full"

finetuning_start = time.time()
# fine-tune the model
model.fit(
    X_train, y_train, num_epochs=100, fine_tuning_type=fine_tuning_type, adapter=adapter
)
finetuning_end = time.time()
print(f"Fine-tuning took {finetuning_end - finetuning_start} seconds")


In [ ]:
prediction_start = time.time()
# predict labels
y_pred = model.predict(X_test)
prediction_end = time.time()
print(f"Prediction took {prediction_end - prediction_start} seconds")
# evaluate performance
print(f"Accuracy on the test set is {np.mean(y_test == y_pred)}")